In [3]:
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, Dropout, Dense, Lambda
import tensorflow.keras.backend as K
from sklearn.metrics import classification_report, confusion_matrix

# -------- HDF5 Data Generator --------
class HDF5DataGenerator(Sequence):
    def __init__(self, file_path, batch_size=256, start=0, end=None, shuffle=True):
        self.file_path = file_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.h5file = h5py.File(self.file_path, 'r')
        self.X = self.h5file['X']
        self.y = self.h5file['y']
        self.start = start
        self.end = end if end is not None else len(self.y)
        self.indices = np.arange(self.start, self.end)
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil((self.end - self.start) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_X = self.X[batch_indices]
        batch_y = self.y[batch_indices]
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# -------- Ross Model Definition --------
def build_ross_model(input_shape=(400, 1)):
    inputs = Input(shape=input_shape)
    x = Conv1D(32, 21, activation="relu")(inputs)
    x = Dropout(0.2)(x)
    x = Conv1D(64, 15, activation="relu")(x)
    x = Dropout(0.2)(x)
    x = Conv1D(128, 11, activation="relu")(x)
    x = Dropout(0.2)(x)
    x = Lambda(lambda t: K.max(t, axis=1))(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    return Model(inputs, outputs)

# -------- Load Dataset Info --------
hdf5_path = 'scsn_p_2000_2017_6sec_0.5r_fm_train.hdf5'

with h5py.File(hdf5_path, 'r') as f:
    total_samples = f['y'].shape[0]

# Use 10% for training, 2% for testing
n_train = int(0.10 * total_samples)
n_test = int(0.02 * total_samples)

train_gen = HDF5DataGenerator(hdf5_path, batch_size=256, start=0, end=n_train)
test_gen = HDF5DataGenerator(hdf5_path, batch_size=256, start=n_train, end=n_train + n_test, shuffle=False)

# -------- Train the Model --------
model = build_ross_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=10, verbose=1)

# -------- Evaluate on Test Set --------
# Manually accumulate predictions and true labels
y_true, y_pred = [], []

for batch_X, batch_y in test_gen:
    y_true.extend(batch_y)
    preds = (model.predict(batch_X) >= 0.5).astype(int).flatten()
    y_pred.extend(preds)

# Report performance
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


ModuleNotFoundError: No module named 'h5py'